# E2 on Kaggle: LLM feature extraction with Qwen2.5-7B-Instruct (AWQ) on vLLM

Serves `Qwen/Qwen2.5-7B-Instruct-AWQ` with vLLM's OpenAI-compatible server, **one server per
GPU** (use T4 x2), and runs the **unchanged** `scripts.llm_extract` against them. Each GPU
handles its own shard of the texts. Schema validation, retries and the disk cache are
identical to any other backend.

**Kaggle setup (right-hand panel)**
1. *Accelerator*: **GPU T4 x2** (T4 x1 also works, at half the speed). *Internet*: **On**.
2. *Add Input*: dataset **`wordsforthewise/lending-club`**.
3. Code: upload `dist/llm-credit-risk-lab-code.zip` (from `python -m scripts.package_code`)
   as a private Kaggle Dataset and add it, **or** set `GIT_URL`.
4. *Resuming* after a timeout: also add the **previous version's output** as an input. Every
   cached answer found there (folders and `e2_llm_outputs.zip`) is merged back in and is not
   re-queried. Its old copy of the code is ignored in favour of the code dataset.

**Run it headless:** *Save Version → Save & Run All (Commit)*. It keeps running with the
browser closed (up to Kaggle's 12 h limit) and keeps `/kaggle/working` as the version output.

**Bring back** `/kaggle/working/e2_llm_outputs.zip`, unzip it into the repo root, then locally:
`python -m scripts.run_experiment --config configs/e2.yaml` (and `e3.yaml`), then
`python -m scripts.make_report`.

> Installing vLLM 0.6.x downgrades torch/numpy and prints many pip "dependency conflict"
> errors about unrelated preinstalled packages (jax, opencv, gradio, ...). They are expected
> and harmless here: every repo step runs in a subprocess. `transformers` is pinned alongside
> vLLM because the image's newer release removed a tokenizer attribute vLLM 0.6.6 uses.

In [ ]:
import glob
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

GIT_URL = ""                                   # or leave "" and upload the repo as a dataset
MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"         # must equal llm.model in configs/base.yaml
VLLM_SPEC = "vllm==0.6.6.post1"
# vLLM 0.6.6 calls tokenizer.all_special_tokens_extended, which newer transformers removed.
TRANSFORMERS_SPEC = "transformers==4.47.1"
BASE_PORT = 8000                               # GPU i serves on BASE_PORT + i
CONCURRENCY = 64                               # in-flight requests per GPU; vLLM batches them
# JSON mode off: vLLM 0.6.6 + the image's newer xgrammar crash the engine on the first
# response_format request (TokenizerInfo.from_huggingface). Qwen2.5 follows the JSON-only
# instruction well, and output is still pydantic-validated with one retry.
JSON_MODE = "off"
# Kaggle stops a run at 12 h. Stop extraction earlier so the packaging cell still runs;
# every answer is cached as it arrives, so the next version resumes where this one stopped.
MAX_HOURS = 10.5
ON_KAGGLE = Path("/kaggle/input").exists()


def is_previous_output(p):
    # A resumed run also mounts the previous version's output, which contains an old repo copy
    # (/repo/) and the bundle (/e2_llm_outputs/). Code must come from the code dataset.
    return bool({"repo", "e2_llm_outputs"} & set(Path(p).parts))


if ON_KAGGLE:
    REPO = Path("/kaggle/working/repo")
    if GIT_URL:
        if not REPO.exists():
            subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
    else:
        found = [Path(p).parent for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)]
        src = [p for p in found if not is_previous_output(p)]
        print("code candidates:", [str(p) for p in found], "-> using", src[:1])
        assert src, "Code not found: upload the repo as a Kaggle Dataset or set GIT_URL"
        CODE = src[0]

        def top_level_only(d, names):
            # Skip data/ and cache/ at the repo root only: src/data/ is a package we need.
            return {n for n in names if Path(d) == CODE and n in {"data", "cache", ".env"}}

        # Re-sync code on every run so a new dataset version reaches this session, while
        # data/ and cache/ already built in /kaggle/working/repo are left untouched.
        shutil.copytree(CODE, REPO, ignore=top_level_only, dirs_exist_ok=True)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
# 127.0.0.1, not localhost: localhost may resolve to IPv6 (::1) while vLLM listens on IPv4, and
# a proxy from the environment must not intercept local calls. Never print os.environ here:
# Kaggle keeps session tokens in it.
os.environ["NO_PROXY"] = os.environ["no_proxy"] = "localhost,127.0.0.1"

from src.config import load_config
from src.features.llm_extract import LLMSettings

cfg = load_config("configs/e2.yaml")
assert cfg["llm"]["model"] == MODEL, f"configs/base.yaml llm.model is {cfg['llm']['model']!r}, expected {MODEL!r}"
SLUG = LLMSettings(model=MODEL).slug
print("repo:", REPO, "| model:", MODEL, "| cache slug:", SLUG)

In [ ]:
def run(*args):
    """Run a repo CLI step and stream its output (same commands as on a laptop)."""
    print("$ python -m", " ".join(args), flush=True)
    subprocess.run([sys.executable, "-m", *args], check=True)


def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if Path(d).exists() else 0


# Resume: merge every cached answer found in the inputs. A previous version's output can hold
# the answers twice, as a folder and inside e2_llm_outputs.zip, and Kaggle may truncate a
# folder with very many files, so neither source is trusted alone. Keys are content hashes:
# a file with the same name is the same answer, so only missing files are copied.
import zipfile

dst = REPO / "cache" / "llm" / SLUG
before = n_files(dst)
have = {f for _, _, fs in os.walk(dst) for f in fs}
for src_dir in glob.glob(f"/kaggle/input/**/cache/llm/{SLUG}", recursive=True):
    added = 0
    for d, _, fs in os.walk(src_dir):
        for f in fs:
            if f.endswith(".json") and f not in have:
                out = dst / f[:2] / f
                out.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(Path(d) / f, out)
                have.add(f)
                added += 1
    print(f"folder {src_dir}: +{added:,}")
marker = f"cache/llm/{SLUG}/"


def restore_from_zip(zf):
    """Copy missing answers out of a zip, wherever the cache sits inside it. Kaggle packs an
    output with many files into one _output_.zip, which can itself contain e2_llm_outputs.zip,
    so zips inside the zip are searched too."""
    import io

    added = 0
    for m in zf.namelist():
        name = Path(m).name
        if m.endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(zf.read(m))) as inner:
                added += restore_from_zip(inner)
        elif marker in m and name.endswith(".json") and name not in have:
            out = dst / name[:2] / name
            out.parent.mkdir(parents=True, exist_ok=True)
            out.write_bytes(zf.read(m))
            have.add(name)
            added += 1
    return added


for d, _, fs in os.walk("/kaggle/input", followlinks=True):
    for f in fs:
        if f.endswith(".zip") and "lending-club" not in d:
            with zipfile.ZipFile(Path(d) / f) as zf:
                print(f"zip {Path(d) / f}: +{restore_from_zip(zf):,}")
print(f"cached answers: {before:,} -> {n_files(dst):,}")


def package(complete):
    """Zip everything needed to resume or to finish locally. Called every 30 minutes too, so
    an interrupted interactive session still leaves a recent zip in /kaggle/working."""
    tmp_zip = "/kaggle/working/e2_llm_outputs.tmp.zip"
    with zipfile.ZipFile(tmp_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in ["data/processed/llm_features", f"cache/llm/{SLUG}", f"reports/runs/llm_{SLUG}",
                  "data/processed/llm_sample_ids.parquet"]:
            if Path(p).is_file():
                zf.write(p, p)
            for d, _, fs in os.walk(p):  # a timed-out run has only the cache: enough to resume
                for f in fs:
                    if not f.endswith(".tmp"):
                        zf.write(Path(d) / f, (Path(d) / f).as_posix())
        zf.writestr("STATUS.txt", "COMPLETE" if complete else "PARTIAL")
    os.replace(tmp_zip, "/kaggle/working/e2_llm_outputs.zip")
    return "/kaggle/working/e2_llm_outputs.zip"

## 1. Install vLLM and prepare the data

In [ ]:
import importlib.util
from importlib.metadata import PackageNotFoundError, version


def installed(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


# Install the pinned pair whenever either pin is not met (a newer preinstalled transformers
# breaks vLLM 0.6.6). The servers run as subprocesses, so no kernel restart is needed.
if installed("vllm") != VLLM_SPEC.split("==")[1] or installed("transformers") != TRANSFORMERS_SPEC.split("==")[1]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", VLLM_SPEC, TRANSFORMERS_SPEC], check=True)
print("vllm", installed("vllm"), "| transformers", installed("transformers"))
for mod, pkg in [("catboost", "catboost"), ("pyarrow", "pyarrow"), ("yaml", "pyyaml"),
                 ("pydantic", "pydantic"), ("openai", "openai")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
N_GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
print("GPUs:", N_GPUS)

In [ ]:
import re


def find_raw(roots):
    """Walk inputs following symlinks (Kaggle mounts datasets via links; glob ** may skip them)."""
    pat = re.compile(r"accepted_2007_to_2018.*[.]csv([.]gz)?$", re.IGNORECASE)
    hits = []
    for root in roots:
        for d, _, files in os.walk(root, followlinks=True):
            hits += [Path(d) / f for f in files if pat.search(f)]
    return sorted(hits, key=lambda p: (p.suffix == ".gz", str(p)))  # prefer uncompressed


if not Path("data/processed/loans.parquet").exists():
    raw = find_raw(["/kaggle/input", "data/raw"])
    if not raw:
        print("Nothing matching accepted_2007_to_2018*.csv(.gz). Contents of /kaggle/input:")
        for d, _dirs, files in os.walk("/kaggle/input", followlinks=True):
            depth = d.count(os.sep) - "/kaggle/input".count(os.sep)
            if depth <= 3:
                print("  " * depth + (os.path.basename(d) or d) + "/", files[:5])
        raise FileNotFoundError("Add the wordsforthewise/lending-club dataset as an input (Add Input -> Datasets)")
    print("raw file:", raw[0])
    run("scripts.prepare_data", "--config", "configs/base.yaml", "--raw", str(raw[0]))

## 2. Start one vLLM server per GPU (fp16, AWQ 4-bit, shared-prompt caching)

In [ ]:
import urllib.request


def start_server(gpu, prefix_caching=True):
    port = BASE_PORT + gpu
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           "--model", MODEL, "--served-model-name", MODEL,
           "--quantization", "awq", "--dtype", "half",        # T4 has no bf16
           "--max-model-len", "2048",                          # prompt <= ~1.5k tokens incl. a retry turn
           "--gpu-memory-utilization", "0.90",
           "--port", str(port), "--disable-log-requests"]
    if prefix_caching:
        # ~400 of ~520 prompt tokens are the same instructions for every loan: compute their
        # KV cache once and reuse it. Outputs are unchanged; prefill work drops sharply.
        cmd.append("--enable-prefix-caching")
    log = open(f"/tmp/vllm{gpu}.log", "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT,
                            env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)})
    return proc, port


def wait_ready(proc, port, gpu, timeout=1200):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=5)
            return True
        except Exception:
            if proc.poll() is not None:
                return False
            time.sleep(10)
    return False


servers = {}
for gpu in range(N_GPUS):
    servers[gpu] = start_server(gpu)
for gpu, (proc, port) in list(servers.items()):
    if not wait_ready(proc, port, gpu):
        print(f"GPU {gpu}: failed with prefix caching, retrying without it. Log tail:")
        print(open(f"/tmp/vllm{gpu}.log").read()[-1500:])
        proc.terminate()
        servers[gpu] = start_server(gpu, prefix_caching=False)
        if not wait_ready(*servers[gpu], gpu):
            print(open(f"/tmp/vllm{gpu}.log").read()[-5000:])
            raise RuntimeError(f"vLLM server on GPU {gpu} did not start")
    print(f"GPU {gpu}: vLLM ready on port {servers[gpu][1]}")
BASE_URL = f"http://127.0.0.1:{BASE_PORT}/v1"

## 3. Pilot: 20 descriptions with their JSON (instant when resuming: they are cached)

In [ ]:
run("scripts.llm_extract", "--config", "configs/e2.yaml", "--n", "20", "--show",
    "--base-url", BASE_URL, "--concurrency", "16", "--json-mode", JSON_MODE)

## 4. Full sample: one shard per GPU in parallel, then assemble (stops cleanly after MAX_HOURS)

In [ ]:
# Read the vLLM version from metadata: importing vllm in this kernel (which already loaded
# numpy 2) could clash with the numpy 1.26 that vLLM 0.6.x installed.
hardware = f"Kaggle T4, vLLM {version('vllm')}, AWQ 4-bit fp16, prefix caching"
shards = []
for gpu, (_, port) in servers.items():
    cmd = [sys.executable, "-m", "scripts.llm_extract", "--config", "configs/e2.yaml",
           "--base-url", f"http://127.0.0.1:{port}/v1", "--concurrency", str(CONCURRENCY),
           "--json-mode", JSON_MODE, "--shard", f"{gpu}/{N_GPUS}", "--hardware", hardware]
    log = open(f"/tmp/shard{gpu}.log", "w")
    shards.append((gpu, subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT), log))
    print("started shard", gpu, "on port", port)

proc_of = {gpu: p for gpu, p, _ in shards}
deadline = time.time() + MAX_HOURS * 3600
tick = 0
while any(p.poll() is None for _, p, _ in shards) and time.time() < deadline:
    time.sleep(600)
    for gpu, _, _ in shards:
        lines = open(f"/tmp/shard{gpu}.log").read().splitlines()
        progress = [line for line in lines if line.startswith("[llm]")]
        alive = "running" if proc_of[gpu].poll() is None else f"exited {proc_of[gpu].returncode}"
        server = "up" if servers[gpu][0].poll() is None else "DOWN"
        status = progress[-1] if progress else "no progress line yet; log tail: " + " | ".join(lines[-2:])
        print(time.strftime("%H:%M"), f"shard {gpu} ({alive}, server {server}):", status, flush=True)
    print("cached answers:", f"{n_files(REPO / 'cache' / 'llm' / SLUG):,}", flush=True)
    tick += 1
    if ON_KAGGLE and tick % 3 == 0:
        package(False)

COMPLETE = all(p.poll() == 0 for _, p, _ in shards)
for gpu, p, log in shards:
    if p.poll() is None:
        p.terminate()
    log.close()
    if p.returncode not in (0, None, -15):
        print(f"shard {gpu} exited with {p.returncode}:\n", open(f"/tmp/shard{gpu}.log").read()[-3000:])
if COMPLETE:
    # Every answer is cached now: this pass only reads the cache and writes the full feature
    # file, and combines the per-GPU cost records into cost.json.
    run("scripts.llm_extract", "--config", "configs/e2.yaml", "--base-url", BASE_URL,
        "--json-mode", JSON_MODE, "--merge-shard-costs")
else:
    print(f"Not finished within {MAX_HOURS} h. Cached answers are kept: add this version's "
          "output as an input to a new version and run again to finish.")
print("cached answers:", f"{n_files(REPO / 'cache' / 'llm' / SLUG):,}")

## 5. Package outputs for download

In [ ]:
for proc, _ in servers.values():
    proc.terminate()
if ON_KAGGLE:
    print(package(COMPLETE))
    print("COMPLETE" if COMPLETE else "PARTIAL - resume in a new version")